<a href="https://colab.research.google.com/github/mf2056/F20AA/blob/main/DataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [76]:
!pip install pandas textblob vaderSentiment scikit-learn

In [77]:
from google.colab import files
uploaded = files.upload()

Saving youtube_amazon_employee.csv to youtube_amazon_employee (1).csv


In [83]:
import pandas as pd
df = pd.read_csv("youtube_amazon_employee.csv")
df.head()

,text,platform,source
0,Currently working at amazon. I work Front half...,youtube,Lyqo2uJvNO4
1,Going on 8.5 years,youtube,Lyqo2uJvNO4
2,Do they have a union? Jeeze…,youtube,Lyqo2uJvNO4
3,Workers having sex in the bathrooms at amazon.,youtube,Lyqo2uJvNO4
4,I work sort. The day I quit Im fucking the sor...,youtube,Lyqo2uJvNO4


In [84]:
#Basic Cleaning

comment_col = "text"
df = df.dropna(subset=[comment_col])  # Drop rows with missing comments
df[comment_col] = df[comment_col].astype(str).str.strip()

# Remove empty strings
df = df[df[comment_col] != ""]

# Remove duplicates
df = df.drop_duplicates(subset=[comment_col])

# Remove very short comments (less than 3 words)
df["comment_word_count"] = df["text"].astype(str).str.split().str.len()
df = df[df["comment_word_count"] >= 3]

df.shape

(2803, 4)

In [85]:
# Remove spam comments

spam_keywords = [
    "subscribe", "giveaway", "click", "telegram", "whatsapp",
    "contact me", "dm me", "crypto", "bitcoin", "forex", "trading",
    "please like", "anyone watching in", "bit.ly", "goo.gl",
    "link in bio", "visit my site"
]

def remove_spam(text):
    text_lower = text.lower()
    return not any(word in text_lower for word in spam_keywords)

df = df[df[comment_col].apply(remove_spam)]
df.shape

(2787, 4)

In [86]:
# Filtering relevant comments

workplace_keywords = [
    "employee", "worker", "staff", "associate", "warehouse",
    "leadership", "executive", "perks", "insurance",
    "fulfillment", "manager", "management", "boss",
    "hr", "supervisor", "shift", "overtime", "break", "pay",
    "salary", "wage", "benefits", "culture", "toxic", "pressure",
    "workload", "stress", "burnout", "union", "treatment",
    "working", "job", "career", "fired", "hired",
    "workplace", "company", "office", "corporate", "hiring",
    "recruitment", "interview", "promotion", "resignation",
    "quit", "quitting", "layoff",
]

def is_relevant(text):
    text_lower = text.lower()
    return any(word in text_lower for word in workplace_keywords)

df = df[df[comment_col].apply(is_relevant)]
df.shape

(1416, 4)

In [89]:
# Textblob labeling

from textblob import TextBlob

def textblob_polarity(text):
    return TextBlob(text).sentiment.polarity

df['tb_polarity'] = df["text"].apply(textblob_polarity)

def tb_to_label(p):
    if p > 0.05:
        return 1      # positive
    elif p < -0.05:
        return -1     # negative
    else:
        return 0      # neutral

df['tb_label'] = df['tb_polarity'].apply(tb_to_label)
df['tb_label'].value_counts(normalize=True)

,proportion
tb_label,
1,0.435028
0,0.336864
-1,0.228107


In [95]:
# Sentiment Analysis

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    return analyzer.polarity_scores(text)['compound']

df["vader_scores"] = df[comment_col].apply(get_vader_scores)

def vader_to_label(compound):
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

df["vader_label"] = df["vader_scores"].apply(vader_to_label)

df["vader_label"].value_counts(normalize=True)

,proportion
vader_label,
positive,0.525424
negative,0.337571
neutral,0.137006


In [31]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment_label"]
)

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)

train_df["sentiment_label"].value_counts(normalize=True)

Train size: (1124, 9)
Test size: (282, 9)


,proportion
sentiment_label,
positive,0.525801
negative,0.338078
neutral,0.136121
